# Agentic Framework — Returns, Refunds & Recommendation Assistant using LangGraph & LangChain

## Overview

This version of the framework integrates **LangGraph** and **LangChain** to create a modular, orchestrated, and conversational AI system for handling returns, refunds, sentiment analysis, and personalized recommendations.

Goals:

* Orchestrate multiple specialized agents using **LangGraph** (graph-based control flow)
* Use **LangChain** for NLU, reasoning, and integration with LLMs, vector stores, and tools
* Map customer intents to business actions (return/refund/appeasement)
* Perform real-time reasoning and retrieval over product data (from `adidas.csv`)
* Recommend alternatives or handle escalations via contextual memory

---

## Architecture using LangGraph + LangChain

```
+-----------------+        +----------------------+        +----------------------+
|  User / Chat UI | --->  |  FastAPI Gateway API | --->  |  LangGraph Workflow  |
+-----------------+        +----------------------+        +----------+-----------+
                                                        |  Nodes / Agents  |
                                                        +------------------+

Main LangGraph Nodes:
  1. IdentityAgentNode
  2. ContextRetrieverNode
  3. NLUNode
  4. PolicyEngineNode
  5. RecommendationNode
  6. FulfillmentNode
  7. FeedbackNode
```

LangGraph handles agent orchestration through **edges** and **state transitions**, while each agent’s logic is implemented using **LangChain components** (LLMs, retrievers, memory, tools).

---

## Core Components

### 🧠 1. Identity Agent (LangChain Tool)

* Retrieves customer data from database or cache
* If user not recognized → triggers fallback to verification step

```python
from langchain.tools import StructuredTool

class IdentityTool(StructuredTool):
    name = "fetch_user_identity"
    description = "Fetch user identity based on phone, email, or login info"

    def _run(self, phone=None, email=None):
        # query postgres or redis
        ...
```

---

### 🔍 2. Context Retriever Agent

* Fetches order details, return eligibility, and past interactions
* Uses LangChain retriever with SQL or API tool

```python
from langchain.agents import create_sql_agent

order_agent = create_sql_agent(
    llm=llm,
    db=db_connection,
    prefix="You are an assistant retrieving customer order and delivery info."
)
```

---

### 💬 3. NLU Agent Node

* Implements intent detection, sentiment analysis, and reason mapping
* Uses LangChain LLM + text embeddings + classification chain

```python
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["message"],
    template="""
    You are a return assistant. Classify the intent, sentiment, and reason from the message:
    Message: {message}
    Respond as JSON with keys: intent, sentiment, reason.
    """
)
nlu_chain = LLMChain(llm=llm, prompt=prompt)
```

---

### ⚙️ 4. Policy Engine Node

* Decides what to do based on reason, severity, and business rules
* Hybrid: rules-based + LLM reasoning

```python
policy_prompt = PromptTemplate(
    input_variables=["reason", "severity", "order_age"],
    template="""
    You are a return policy engine. Based on the reason {reason} and severity {severity}, decide whether to:
    - Approve return
    - Offer coupon
    - Escalate to human
    Return a JSON policy decision.
    """
)
policy_chain = LLMChain(llm=llm, prompt=policy_prompt)
```

---

### 🛍️ 5. Recommendation Node

* Suggests better-fitting or similar products using `adidas.csv`
* Uses LangChain’s **vectorstore retriever** (FAISS / Milvus)

```python
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(df['prod_text'].tolist(), embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

recommendation_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff"
)
```

---

### 📦 6. Fulfillment Node

* Integrates with warehouse or ERP via REST API (LangChain tool)

```python
from langchain.tools import StructuredTool

class CreateReturnLabelTool(StructuredTool):
    name = "create_return_label"
    description = "Creates a return label and schedules pickup"

    def _run(self, order_id):
        # call external API
        ...
```

---

### 📊 7. Feedback Node

* Logs feedback, decisions, and user reactions
* Uses LangChain’s memory and callback manager for tracking user flow

```python
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(memory_key="chat_history")
```

---

## LangGraph Workflow Example

```python
from langgraph import StateGraph, END

workflow = StateGraph()

workflow.add_node("identity", IdentityTool())
workflow.add_node("context", order_agent)
workflow.add_node("nlu", nlu_chain)
workflow.add_node("policy", policy_chain)
workflow.add_node("recommend", recommendation_chain)
workflow.add_node("fulfill", CreateReturnLabelTool())

workflow.add_edge("identity", "context")
workflow.add_edge("context", "nlu")
workflow.add_edge("nlu", "policy")
workflow.add_edge("policy", "recommend")
workflow.add_edge("recommend", "fulfill")
workflow.add_edge("fulfill", END)

app = workflow.compile()
response = app.invoke({"message": "I want to return my shoes, they don’t fit."})
print(response)
```

---

## Suggested LLM Stack

* **LLM**: GPT-4, Claude, or Mixtral (for reasoning)
* **Embeddings**: Sentence Transformers (MiniLM / MPNet)
* **Vector DB**: FAISS (local) or Milvus (scalable)
* **Frameworks**: LangGraph + LangChain + FastAPI + Postgres
* **Optional Tools**: Redis for caching, Prometheus for metrics

---

## Sample Flow

**User:** “Hey, I want to return my shoes, they don’t fit.”
**Agentic flow:**

1. Identity → finds user
2. Context → fetches order details
3. NLU → detects intent: return_request; reason: fit_issue; sentiment: neutral
4. Policy → decides: approve return, no coupon
5. Recommender → finds similar shoes with right size
6. Fulfillment → triggers pickup label
7. Feedback → logs interaction

---

## Next Steps

1. Implement each node as a LangChain tool or chain.
2. Configure the LangGraph transitions and error edges.
3. Integrate FAISS retriever with `adidas.csv` embeddings.
4. Deploy on FastAPI or Streamlit for testing.
5. Add memory persistence and user feedback loop.

---

This architecture provides a **production-grade agentic flow** using **LangGraph** for orchestration and **LangChain** for intelligent reasoning, tool integration, and retrieval.
